# 04 The coverage gap between the MSitAE report tables and the CQI open data
The footnote says the MSitAE report tables account for revisions and may differ slightly from CQI. I test what actually differs. CQI publishes no four-hour measure, so the comparison is on attendances (AEQI012) and on the scope of providers; the four-hour effect is **my calculation**, not a published figure.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
m = pd.read_parquet(c.SILVER / 'msitae_monthly_provider.parquet')
cqi = pd.read_parquet(c.SILVER / 'cqi_long.parquet')
t11 = pd.read_parquet(c.SILVER / 'msitae_table11_dept_type.parquet').set_index('dept_type')
t3 = pd.read_parquet(c.SILVER / 'msitae_table3_four_hour.parquet').set_index('year')
m['att'] = m[c.ATT_COLS].sum(axis=1) + m[c.BOOKED_COLS].sum(axis=1)
m['over4'] = m[c.OVER4_COLS].sum(axis=1) + m[c.OVER4_BOOKED_COLS].sum(axis=1)   # booked included, as the gate showed
cq = cqi[(cqi.measure_id == 'AEQI012') & (cqi.org_code != 'ENG')].rename(columns={'measure_value': 'cqi_att'})[['month', 'org_code', 'cqi_att']]
cq = cq[cq.cqi_att > 0]

## 1. Monthly totals: sitrep against CQI

In [3]:
sitrep_m = m.groupby('month').att.sum()
cqi_m = cqi[(cqi.measure_id == 'AEQI012') & (cqi.org_code == 'ENG')].set_index('month').measure_value
monthly = pd.DataFrame({'sitrep_all_providers': sitrep_m, 'cqi_msitae': cqi_m})
monthly['omitted_by_cqi'] = monthly.sitrep_all_providers - monthly.cqi_msitae
monthly['omitted_pct'] = 100 * monthly.omitted_by_cqi / monthly.sitrep_all_providers
print(monthly.round(3).to_string()); print('annual omitted:', int(monthly.omitted_by_cqi.sum()), f'({100*monthly.omitted_by_cqi.sum()/monthly.sitrep_all_providers.sum():.2f}%)')

         sitrep_all_providers     cqi_msitae  omitted_by_cqi  omitted_pct
month                                                                    
2025-04               2297421 2,223,533.0000     73,888.0000       3.2160
2025-05               2397529 2,316,892.0000     80,637.0000       3.3630
2025-06               2351581 2,273,248.0000     78,333.0000       3.3310
2025-07               2408866 2,331,432.0000     77,434.0000       3.2150
2025-08               2266937 2,193,323.0000     73,614.0000       3.2470
2025-09               2310222 2,235,469.0000     74,753.0000       3.2360
2025-10               2401266 2,334,810.0000     66,456.0000       2.7680
2025-11               2347046 2,283,711.0000     63,335.0000       2.6980
2025-12               2326139 2,262,399.0000     63,740.0000       2.7400
2026-01               2319296 2,258,774.0000     60,522.0000       2.6090
2026-02               2117450 2,059,276.0000     58,174.0000       2.7470
2026-03               2432272 2,366,79

## 2. Where CQI overlaps with the sitrep, how different is it?
I test every provider-month, not just the totals.

In [4]:
j = m[['month', 'org_code', 'org_name', 'att', 'over4']].merge(cq, on=['month', 'org_code'], how='left')
shared = j[j.cqi_att.notna()]
print('provider-months in sitrep with activity:', int((j.att > 0).sum()), '| present in CQI:', len(shared))
print('exact matches among shared provider-months:', int((shared.att == shared.cqi_att).sum()), 'of', len(shared))
print('largest absolute difference among shared:', int((shared.att - shared.cqi_att).abs().max()))

provider-months in sitrep with activity: 2271 | present in CQI: 1838
exact matches among shared provider-months: 1838 of 1838
largest absolute difference among shared: 0


## 3. The providers CQI leaves out

In [5]:
annual = j.groupby(['org_code']).agg(org_name=('org_name', 'first'), att=('att', 'sum'), over4=('over4', 'sum'), in_cqi=('cqi_att', lambda s: s.notna().any()))
omitted = annual[(~annual.in_cqi) & (annual.att > 0)].copy()
omitted['four_hour_pct'] = 100 * (1 - omitted.over4 / omitted.att)
omitted = omitted.sort_values('att', ascending=False).reset_index()
print(len(omitted), 'providers with activity omitted; attendances', int(omitted.att.sum()))
omitted.head(12)

41 providers with activity omitted; attendances 836365


,org_code,org_name,att,over4,in_cqi,four_hour_pct
0,Y02615,SOUTH BIRMINGHAM GP WALK IN CENTRE,63151,0,False,100.0000
1,NTP11,PRACTICE PLUS GROUP HOSPITAL - SOUTHAMPTON,62759,1311,False,97.9111
2,Y02147,URGENT CARE CENTRE (QMS),49604,734,False,98.5203
3,NNF09,BRANSHOLME HEALTH CENTRE,44480,5099,False,88.5364
4,NL7,ASSURA VERTIS URGENT CARE CENTRES (BIRMINGHAM),38316,0,False,100.0000
5,NQTE4,SUMMERFIELD URGENT CARE CENTRE,36424,0,False,100.0000
6,NNF94,EAST RIDING COMMUNITY HOSPITAL,33881,2094,False,93.8195
7,G0Q0L,BARKING HOSPITAL UTC,33460,216,False,99.3545
8,K6K0R,HAROLD WOOD POLYCLINIC UTC,32592,168,False,99.4845
9,AQM01,FAVERSHAM MEDICAL PRACTICE MIU,32511,0,False,100.0000


## 4. Is it a code mismatch?
CQI also lists organisations with ECDS attendances but no MSitAE count. I test whether the omitted sitrep codes are children of those codes by prefix. I expect a partial match at best, and I report what I find.

In [6]:
w = cqi[cqi.org_code != 'ENG'].pivot_table(index='org_code', columns='measure_id', values='measure_value', aggfunc='sum')[['AEQI011', 'AEQI012']].fillna(0)
orphans = w[(w.AEQI012 == 0) & (w.AEQI011 > 0)]
print('CQI orgs:', len(w), '| all with ECDS > 0:', bool((w.AEQI011 > 0).all()), '| ECDS-only orgs:', len(orphans), 'with ECDS attendances', int(orphans.AEQI011.sum()))
def parent(code):
    for n in (5, 4, 3):
        if code[:n] != code and code[:n] in orphans.index:
            return code[:n]
omitted['cqi_parent'] = omitted.org_code.map(parent)
hit = omitted[omitted.cqi_parent.notna()]
print(f'{len(hit)} of {len(omitted)} omitted providers share a code prefix with a CQI ECDS-only organisation, covering {int(hit.att.sum()):,} of {int(omitted.att.sum()):,} attendances')
cmp_ = hit.groupby('cqi_parent').att.sum().rename('sitrep_att').to_frame().join(orphans.AEQI011.rename('cqi_ecds'))
cmp_['ecds_over_sitrep'] = cmp_.cqi_ecds / cmp_.sitrep_att
cmp_

CQI orgs: 185 | all with ECDS > 0: True | ECDS-only orgs: 31 with ECDS attendances 1574645
8 of 41 omitted providers share a code prefix with a CQI ECDS-only organisation, covering 207,626 of 836,365 attendances


,sitrep_att,cqi_ecds,ecds_over_sitrep
cqi_parent,,,
ARN,29845,"7,895.0000",0.2645
NNF,116891,"162,435.0000",1.3896
NQT,60890,"30,330.0000",0.4981


## 5. What the omission does to the numbers people quote

In [7]:
inc_orgs = set(cq.org_code)
m['in_cqi'] = m.org_code.isin(inc_orgs)
def perf(df): return 100 * (1 - df.over4.sum() / df.att.sum())
scope = pd.DataFrame({'all_providers': m.groupby('month').apply(perf, include_groups=False),
                      'cqi_providers_only': m[m.in_cqi].groupby('month').apply(perf, include_groups=False)})
scope['difference_pp'] = scope.cqi_providers_only - scope.all_providers
annual_all, annual_cqi = perf(m), perf(m[m.in_cqi])
print(f'annual four-hour %: all providers {annual_all:.4f} (published {100*t3.loc["2025-26","pct_le4"]:.4f}) | CQI providers only {annual_cqi:.4f} | difference {annual_cqi-annual_all:.3f} pp')
ecds = cqi[(cqi.measure_id == 'AEQI011') & (cqi.org_code == 'ENG')].measure_value.sum()
cov = pd.DataFrame({'source': ['CQI (AEQI011 / AEQI012, my calculation)', 'Report Table 11'],
                    'ecds': [ecds, t11.ecds.iloc[3]], 'msitae': [monthly.cqi_msitae.sum(), t11.msitae.iloc[3]]})
cov['ecds_pct_of_msitae'] = 100 * cov.ecds / cov.msitae
scope.round(3), cov

annual four-hour %: all providers 74.9288 (published 74.9288) | CQI providers only 74.2217 | difference -0.707 pp


(         all_providers  cqi_providers_only  difference_pp
 month                                                    
 2025-04        74.7840             74.0360        -0.7470
 2025-05        75.4020             74.6400        -0.7620
 2025-06        75.5560             74.8130        -0.7430
 2025-07        76.4080             75.7150        -0.6930
 2025-08        75.8760             75.1440        -0.7320
 2025-09        75.0500             74.3130        -0.7370
 2025-10        74.1860             73.4860        -0.6990
 2025-11        74.2170             73.5430        -0.6740
 2025-12        73.8420             73.1480        -0.6940
 2026-01        72.4510             71.7420        -0.7090
 2026-02        74.0550             73.3780        -0.6770
 2026-03        77.1110             76.5130        -0.5980,
                                     source            ecds          msitae  ecds_pct_of_msitae
 0  CQI (AEQI011 / AEQI012, my calculation) 26,631,296.0000 27,139,660.0000  

In [8]:
monthly.reset_index().to_parquet(c.GOLD / 'coverage_gap_monthly.parquet', index=False)
omitted.to_parquet(c.GOLD / 'omitted_providers.parquet', index=False)
scope.reset_index().to_parquet(c.GOLD / 'four_hour_scope_effect_monthly.parquet', index=False)
cov.to_parquet(c.GOLD / 'ecds_coverage_ratio.parquet', index=False)

## The chart
One axis, two lines, one reference rule. The y-axis is truncated to make a 0.7-point gap visible, so the axis range is labelled in the subtitle.

In [9]:
BLUE, ORANGE, INK, MUTED, GRID, SURFACE = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
fig, ax = plt.subplots(figsize=(9, 5), facecolor=SURFACE); ax.set_facecolor(SURFACE)
x = np.arange(len(scope)); labels = pd.to_datetime(scope.index).strftime('%b\n%y')
ax.plot(x, scope.all_providers, color=BLUE, lw=2, marker='o', ms=5, mec=SURFACE, mew=1.5, label=f'All {m[m.att>0].org_code.nunique()} providers (report-table scope)')
ax.plot(x, scope.cqi_providers_only, color=ORANGE, lw=2, ls='--', marker='s', ms=5, mec=SURFACE, mew=1.5, label=f'{len(inc_orgs)} providers present in CQI')
ax.axhline(74.9, color=MUTED, lw=1, ls=':'); ax.text(8.5, 74.9 + 0.1, 'annual 74.9% (report tables)', ha='center', va='bottom', fontsize=9, color=MUTED)
ax.text(x[-1] + 0.15, scope.all_providers.iloc[-1], 'all providers', color=INK, va='center', fontsize=9)
ax.text(x[-1] + 0.15, scope.cqi_providers_only.iloc[-1] - 0.1, 'CQI providers', color=INK, va='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9, color=MUTED); ax.set_ylim(71, 77.5); ax.set_xlim(-0.4, len(x) + 0.9)
ax.set_ylabel('% of attendances within four hours (axis starts at 71%)', fontsize=9, color=MUTED); ax.tick_params(colors=MUTED, length=0)
ax.grid(axis='y', color=GRID, lw=0.8); [s.set_visible(False) for s in ax.spines.values()]
ax.legend(frameon=False, loc='lower left', fontsize=9, labelcolor=INK)
fig.text(0.07, 0.96, 'Leaving out urgent care sites lowers four-hour performance\nby about 0.7 points in every month', fontsize=12, fontweight='bold', color=INK, ha='left', va='top')
fig.text(0.07, 0.885, 'England A&E 2025-26. CQI has no four-hour measure; the orange line is my recalculation on CQI\'s provider set.', fontsize=9, color=MUTED, ha='left', va='top')
fig.subplots_adjust(top=0.82, left=0.1, right=0.97, bottom=0.12)
fig.savefig(c.GOLD / 'charts' / 'four_hour_scope_effect.png', dpi=160, facecolor=SURFACE)
plt.show()

/var/folders/b8/2010kgtx6cb7kd31k135xwy40000gn/T/ipykernel_15190/201913457.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
